In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import warnings
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("\n✓ Imports OK")

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB

✓ Imports OK


In [ ]:
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("⚠ No GPU — Runtime → Change runtime type → T4 GPU")

print("\n✓ Imports OK")

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB

✓ Imports OK


In [ ]:
DRIVE_DIR  = Path("/content/drive/MyDrive")

DATA_DIR   = DRIVE_DIR / "data_4x"
OUT_DIR    = DRIVE_DIR / "aneo_output"
MODEL_DIR  = DRIVE_DIR / "aneo_models"
TENSOR_DIR = DRIVE_DIR / "aneo_tensors"
FIG_DIR    = DRIVE_DIR / "aneo_figures"

for d in [OUT_DIR, MODEL_DIR, TENSOR_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"split.json exists       : {(TENSOR_DIR/'split.json').exists()}")
print(f"normalization stats     : {(OUT_DIR/'normalization_stats.csv').exists()}")
print(f"grid_features.npy       : {(TENSOR_DIR/'grid_features.npy').exists()}")
tensor_files = sorted(TENSOR_DIR.glob("*.npy"))
print(f"Tensor files (.npy)     : {len(tensor_files)}")
print(f"\n✓ Paths OK")

split.json exists       : True
normalization stats     : True
grid_features.npy       : True
Tensor files (.npy)     : 241

✓ Paths OK


In [ ]:
# ── ONE-TIME: create grid_features.npy ───────────────────────────────────────
if (TENSOR_DIR / "grid_features.npy").exists():
    print("grid_features.npy already exists on Drive — skipping")
else:
    grid = pd.read_csv(DATA_DIR / "aneo_grid_points.csv")
    grid = grid.sort_values(['iy', 'ix']).reset_index(drop=True)
    NX_g, NY_g = 1000, 500

    def normalize(arr):
        return (arr - arr.mean()) / (arr.std() + 1e-8)

    alt_grid     = normalize(grid['z_terrain_m'].values).reshape(NY_g, NX_g)
    zrel_grid    = normalize(grid['z_rel_m'].values).reshape(NY_g, NX_g)
    z0_grid      = normalize(grid['z0_m'].values).reshape(NY_g, NX_g)
    slope_grid   = normalize(grid['slope_deg'].values).reshape(NY_g, NX_g)
    aspect_rad   = np.deg2rad(grid['aspect_deg'].values)
    asp_sin_grid = np.sin(aspect_rad).reshape(NY_g, NX_g).astype(np.float32)
    asp_cos_grid = np.cos(aspect_rad).reshape(NY_g, NX_g).astype(np.float32)
    x_grid       = normalize(grid['x_m'].values).reshape(NY_g, NX_g)
    y_grid       = normalize(grid['y_m'].values).reshape(NY_g, NX_g)

    grid_features = np.stack([
        alt_grid, zrel_grid, z0_grid, slope_grid,
        asp_sin_grid, asp_cos_grid, x_grid, y_grid,
    ], axis=-1).astype(np.float32)

    np.save(TENSOR_DIR / "grid_features.npy", grid_features)
    print(f"grid_features shape : {grid_features.shape}")
    print(f"✓ grid_features.npy saved")

grid_features.npy already exists on Drive — skipping


In [ ]:
with open(TENSOR_DIR / "split.json", "r") as f:
    split = json.load(f)

TRAIN_DAYS  = split["train_days"]
VAL_DAYS    = split["val_days"]
TEST_DAYS   = split["test_days"]
HEURES      = split["heures"]
NX          = split["nx"]        # 1000
NY          = split["ny"]        # 500
N_INPUT     = split["n_input"]   # 8
N_TARGET    = split["n_target"]  # 3
INPUT_COLS  = split["input_cols"]
TARGET_COLS = split["target_cols"]

norm_stats = pd.read_csv(OUT_DIR / "normalization_stats.csv", index_col=0)

STD_U = norm_stats.loc["wind_u_m_s", "std"]
STD_V = norm_stats.loc["wind_v_m_s", "std"]
STD_W = norm_stats.loc["wind_w_m_s", "std"]
MU_U  = norm_stats.loc["wind_u_m_s", "mean"]
MU_V  = norm_stats.loc["wind_v_m_s", "mean"]
MU_W  = norm_stats.loc["wind_w_m_s", "mean"]

# Input channels
N_GRID_FEATURES     = 8   # z_terrain, z_rel, z0, slope, asp_sin, asp_cos, x, y
N_TEMPORAL_FEATURES = 2   # hour_sin, hour_cos
N_INPUT_EFFECTIVE   = N_INPUT + N_GRID_FEATURES + N_TEMPORAL_FEATURES  # 18

print("Configuration :\n")
print(f"  Train : {len(TRAIN_DAYS)} days → {TRAIN_DAYS}")
print(f"  Val   : {len(VAL_DAYS)} days → {VAL_DAYS}")
print(f"  Test  : {len(TEST_DAYS)} days → {TEST_DAYS}")
print(f"\n  Grid     : {NX} x {NY}")
print(f"  N_INPUT (effective) : {N_INPUT_EFFECTIVE}")
print(f"  N_TARGET            : {N_TARGET}")
print(f"\n  STD u,v,w : {STD_U:.4f}, {STD_V:.4f}, {STD_W:.4f}")
print(f"  MU  u,v,w : {MU_U:.4f}, {MU_V:.4f}, {MU_W:.4f}")
print(f"\n✓ Config OK")

Configuration :

  Train : 19 days → [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
  Val   : 5 days → [20, 21, 22, 24, 25]
  Test  : 6 days → [23, 26, 27, 28, 29, 30]

  Grid     : 1000 x 500
  N_INPUT (effective) : 18
  N_TARGET            : 3

  STD u,v,w : 2.0169, 2.1665, 0.0621
  MU  u,v,w : 1.5670, 4.1370, -0.0272

✓ Config OK


In [ ]:
class ANEODataset(Dataset):
    """
    18-channel dataset: 8 original + 8 grid + 2 temporal.
    No wind rotation augmentation (physically inconsistent
    with static terrain features).
    """
    def __init__(self, days, heures, tensor_dir, augment=False):
        self.tensor_dir = Path(tensor_dir)
        self.augment    = augment
        self.samples    = []

        for day in days:
            for h in heures:
                x_path = self.tensor_dir / f"day_{day:03d}_h{h:02d}_X.npy"
                y_path = self.tensor_dir / f"day_{day:03d}_h{h:02d}_Y.npy"
                if x_path.exists() and y_path.exists():
                    self.samples.append((x_path, y_path, day, h))

        # Load static grid features once — (8, NY, NX)
        gf = np.load(self.tensor_dir / "grid_features.npy")
        self.grid_features = torch.from_numpy(
            gf.transpose(2, 0, 1).astype(np.float32))

        print(f"  Dataset : {len(self.samples)} snapshots "
              f"({'aug ON' if augment else 'aug OFF'})  "
              f"18 channels")

    def __len__(self):
        return len(self.samples)

    def augment_sample(self, X, Y):
        X, Y = X.clone(), Y.clone()
        if torch.rand(1) > 0.5:
            scale = torch.FloatTensor(1).uniform_(0.8, 1.2).item()
            X[4:7] *= scale
            Y      *= scale
        if torch.rand(1) > 0.5:
            X[:8] += torch.randn_like(X[:8]) * 0.02
        return X, Y

    def __getitem__(self, idx):
        x_path, y_path, day, h = self.samples[idx]

        X = torch.from_numpy(
            np.load(x_path).astype(np.float32).transpose(2, 0, 1))
        Y = torch.from_numpy(
            np.load(y_path).astype(np.float32).transpose(2, 0, 1))

        if self.augment:
            X, Y = self.augment_sample(X, Y)

        # Add grid features (8, NY, NX)
        X = torch.cat([X, self.grid_features], dim=0)  # (16, NY, NX)

        # Add temporal encoding (2, NY, NX)
        _, H, W = X.shape
        hour_sin = float(np.sin(2 * np.pi * h / 24))
        hour_cos = float(np.cos(2 * np.pi * h / 24))
        sin_ch = torch.full((1, H, W), hour_sin, dtype=torch.float32)
        cos_ch = torch.full((1, H, W), hour_cos, dtype=torch.float32)
        X = torch.cat([X, sin_ch, cos_ch], dim=0)  # (18, NY, NX)

        return X, Y, day, h


print("✓ ANEODataset defined")

✓ ANEODataset defined


In [ ]:
BATCH_SIZE = 1

train_dataset = ANEODataset(TRAIN_DAYS + VAL_DAYS, HEURES,
                            TENSOR_DIR, augment=True)
val_dataset   = ANEODataset(VAL_DAYS,   HEURES,
                            TENSOR_DIR, augment=False)
test_dataset  = ANEODataset(TEST_DAYS,  HEURES,
                            TENSOR_DIR, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=1,
                          shuffle=False, num_workers=2)

print(f"\nBatch size     : {BATCH_SIZE}")
print(f"Train batches  : {len(train_loader)}")
print(f"Val   batches  : {len(val_loader)}")
print(f"Test  batches  : {len(test_loader)}")

X_b, Y_b, _, _ = next(iter(train_loader))
print(f"\nX : {X_b.shape}  ← should be [1, 18, 500, 1000]")
print(f"Y : {Y_b.shape}  ← should be [1,  3, 500, 1000]")

assert X_b.shape[1] == N_INPUT_EFFECTIVE, \
    f"X channels={X_b.shape[1]} but expected {N_INPUT_EFFECTIVE}"

print(f"\n✓ Dataloaders ready")

  Dataset : 96 snapshots (aug ON)  18 channels
  Dataset : 20 snapshots (aug OFF)  18 channels
  Dataset : 24 snapshots (aug OFF)  18 channels

Batch size     : 1
Train batches  : 96
Val   batches  : 20
Test  batches  : 24

X : torch.Size([1, 18, 500, 1000])  ← should be [1, 18, 500, 1000]
Y : torch.Size([1, 3, 500, 1000])  ← should be [1,  3, 500, 1000]

✓ Dataloaders ready


In [ ]:
class PatchEmbed(nn.Module):
    """
    Splits 500×1000 spatial field into non-overlapping patches.
    patch_size=25 → 20×40 = 800 tokens per field.
    Each patch: 18 channels × 25×25 pixels → embed_dim vector.
    """
    def __init__(self, in_channels=18, patch_size=25, embed_dim=192):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)                          # (B, embed_dim, H/P, W/P)
        B, E, Hp, Wp = x.shape
        x = x.flatten(2).transpose(1, 2)          # (B, Hp*Wp, embed_dim)
        return self.norm(x), Hp, Wp


class TransformerBlock(nn.Module):
    """
    Standard ViT block:
    Pre-norm → Multi-Head Self-Attention → residual
    Pre-norm → MLP (FFN) → residual
    """
    def __init__(self, embed_dim=192, num_heads=6,
                 mlp_ratio=4, dropout=0.05):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = nn.MultiheadAttention(
            embed_dim, num_heads,
            dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden = int(embed_dim * mlp_ratio)
        self.mlp   = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        n = self.norm1(x)
        attn_out, _ = self.attn(n, n, n)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class WindViT(nn.Module):
    """
    Vision Transformer for 3D wind field reconstruction.

    Architecture:
      1. PatchEmbed  : 18ch × 500×1000 → 800 tokens of dim 192
      2. Pos embed   : learnable position encoding per token
      3. Transformer : 6 blocks of self-attention + FFN
      4. Reshape     : 800 tokens → (192, 20, 40) feature map
      5. Upsample    : bilinear interpolation → (192, 500, 1000)
                       guarantees exact output size regardless of patch size
      6. Refine      : Conv layers to recover fine spatial detail
      7. Heads       : separate Conv heads for u, v, w

    Key fix vs previous version:
      ConvTranspose2d upsampling was producing wrong output dimensions.
      Bilinear F.interpolate(size=(NY, NX)) is guaranteed correct.
    """
    def __init__(self, in_channels=18, patch_size=25,
                 embed_dim=192, depth=6, num_heads=6,
                 mlp_ratio=4, dropout=0.05):
        super().__init__()

        self.patch_size = patch_size
        self.embed_dim  = embed_dim

        # ── Patch embedding ───────────────────────────────────────────────────
        self.patch_embed = PatchEmbed(in_channels, patch_size, embed_dim)

        # ── Positional embedding ──────────────────────────────────────────────
        n_patches_h = NY  // patch_size   # 500/25 = 20
        n_patches_w = NX  // patch_size   # 1000/25 = 40
        n_tokens    = n_patches_h * n_patches_w  # 800

        self.pos_embed = nn.Parameter(
            torch.zeros(1, n_tokens, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.pos_drop = nn.Dropout(dropout)

        # ── Transformer blocks ────────────────────────────────────────────────
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

        # ── Decoder: bilinear upsample + conv refinement ──────────────────────
        # F.interpolate(size=(NY, NX)) guarantees exact (500, 1000) output
        # regardless of patch size or grid dimensions — no shape mismatch risk
        self.refine = nn.Sequential(
            nn.Conv2d(embed_dim, 128, kernel_size=3, padding=1),
            nn.InstanceNorm2d(128, affine=True),
            nn.GELU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.InstanceNorm2d(128, affine=True),
            nn.GELU(),
            nn.Conv2d(128, 64,  kernel_size=3, padding=1),
            nn.InstanceNorm2d(64, affine=True),
            nn.GELU(),
            nn.Conv2d(64,  64,  kernel_size=3, padding=1),
            nn.GELU(),
        )

        # ── Separate output heads per wind component ──────────────────────────
        def make_head():
            return nn.Sequential(
                nn.Conv2d(64, 32, kernel_size=3, padding=1), nn.GELU(),
                nn.Conv2d(32, 16, kernel_size=3, padding=1), nn.GELU(),
                nn.Conv2d(16, 1,  kernel_size=1),
            )

        self.head_u = make_head()
        self.head_v = make_head()
        self.head_w = make_head()

    def forward(self, x):
        B = x.shape[0]

        # ── Tokenize ──────────────────────────────────────────────────────────
        tokens, Hp, Wp = self.patch_embed(x)     # (B, 800, 192)
        tokens = self.pos_drop(tokens + self.pos_embed)

        # ── Self-attention blocks ─────────────────────────────────────────────
        for block in self.blocks:
            tokens = block(tokens)
        tokens = self.norm(tokens)               # (B, 800, 192)

        # ── Reshape to spatial feature map ────────────────────────────────────
        feat = tokens.transpose(1, 2).reshape(
            B, self.embed_dim, Hp, Wp)           # (B, 192, 20, 40)

        # ── Bilinear upsample to exact original resolution ────────────────────
        # This is the key fix: guaranteed (500, 1000) output
        feat = F.interpolate(
            feat.float(),
            size=(NY, NX),
            mode='bilinear',
            align_corners=False)                 # (B, 192, 500, 1000)

        # ── Convolutional refinement ──────────────────────────────────────────
        feat = self.refine(feat)                 # (B, 64, 500, 1000)

        # ── Separate component predictions ────────────────────────────────────
        u = self.head_u(feat)                    # (B, 1, 500, 1000)
        v = self.head_v(feat)                    # (B, 1, 500, 1000)
        w = self.head_w(feat)                    # (B, 1, 500, 1000)

        return torch.cat([u, v, w], dim=1)       # (B, 3, 500, 1000)


# ── Instantiate model ─────────────────────────────────────────────────────────
model = WindViT(
    in_channels = N_INPUT_EFFECTIVE,  # 18
    patch_size  = 25,                 # 500/25=20 × 1000/25=40 = 800 tokens
    embed_dim   = 192,
    depth       = 6,
    num_heads   = 6,
    mlp_ratio   = 4,
    dropout     = 0.05,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"WindViT Model (fixed decoder) :")
print(f"  in_channels : {N_INPUT_EFFECTIVE}")
print(f"  patch_size  : 25×25")
print(f"  tokens      : {NY//25}×{NX//25} = {(NY//25)*(NX//25)} per field")
print(f"  embed_dim   : 192")
print(f"  depth       : 6  transformer blocks")
print(f"  num_heads   : 6  attention heads")
print(f"  mlp_ratio   : 4")
print(f"  decoder     : F.interpolate(500,1000) + Conv refine")
print(f"  heads       : separate u / v / w")
print(f"  parameters  : {n_params:,}")
print(f"  device      : {device}")
print(f"\n✓ WindViT ready")

WindViT Model (fixed decoder) :
  in_channels : 18
  patch_size  : 25×25
  tokens      : 20×40 = 800 per field
  embed_dim   : 192
  depth       : 6  transformer blocks
  num_heads   : 6  attention heads
  mlp_ratio   : 4
  decoder     : F.interpolate(500,1000) + Conv refine
  heads       : separate u / v / w
  parameters  : 5,533,315
  device      : cuda

✓ WindViT ready


In [ ]:
class PhysicsConstrainedLoss(nn.Module):
    def __init__(self, lambda_mass=0.3, lambda_bc=0.05,
                 lambda_smooth=0.01, lambda_ri=0.02, dx=50.0):
        super().__init__()
        self.lambda_mass   = lambda_mass
        self.lambda_bc     = lambda_bc
        self.lambda_smooth = lambda_smooth
        self.lambda_ri     = lambda_ri
        self.dx            = dx

    def forward(self, pred, target, rho, phys_weight=1.0):
        L_data = F.mse_loss(pred, target)

        u_pred = pred[:, 0:1, :, :]
        v_pred = pred[:, 1:2, :, :]

        rho_u    = rho * u_pred
        rho_v    = rho * v_pred
        drhou_dx = (rho_u[:,:,:,2:] - rho_u[:,:,:,:-2]) / (2*self.dx)
        drhov_dy = (rho_v[:,:,2:,:] - rho_v[:,:,:-2,:]) / (2*self.dx)
        min_y    = min(drhou_dx.shape[2], drhov_dy.shape[2])
        min_x    = min(drhou_dx.shape[3], drhov_dy.shape[3])
        div      = (drhou_dx[:,:,:min_y,:min_x] +
                    drhov_dy[:,:,:min_y,:min_x])
        L_mass   = (div**2).mean()

        w_pred = pred[:, 2:3, :, :]
        L_bc   = (w_pred[:,:,0,:]**2).mean()

        grad_u_x = pred[:,0:1,:,1:] - pred[:,0:1,:,:-1]
        grad_u_y = pred[:,0:1,1:,:] - pred[:,0:1,:-1,:]
        grad_v_x = pred[:,1:2,:,1:] - pred[:,1:2,:,:-1]
        grad_v_y = pred[:,1:2,1:,:] - pred[:,1:2,:-1,:]
        L_smooth = ((grad_u_x**2).mean() + (grad_u_y**2).mean() +
                    (grad_v_x**2).mean() + (grad_v_y**2).mean())

        du_dx  = (u_pred[:,:,:,1:] - u_pred[:,:,:,:-1]) / self.dx
        dv_dy  = (v_pred[:,:,1:,:] - v_pred[:,:,:-1,:]) / self.dx
        L_ri   = (F.relu(du_dx**2-1.0).mean() +
                  F.relu(dv_dy**2-1.0).mean())

        phys_term = (self.lambda_mass   * L_mass +
                     self.lambda_bc     * L_bc +
                     self.lambda_smooth * L_smooth +
                     self.lambda_ri     * L_ri)

        return L_data + phys_weight * phys_term, L_data, L_mass, L_bc


criterion = PhysicsConstrainedLoss(
    lambda_mass=0.3, lambda_bc=0.05,
    lambda_smooth=0.01, lambda_ri=0.02, dx=50.0)

print("✓ Physics loss ready")

✓ Physics loss ready


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr           = 3e-4,
    weight_decay = 1e-4,
    betas        = (0.9, 0.999),
)

N_EPOCHS      = 150
WARMUP_EPOCHS = 15
EARLY_STOP    = 25
ACCUM_STEPS   = 4

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS, eta_min=1e-6
)

scaler = GradScaler('cuda')

print("Training config :")
print(f"  N_EPOCHS      : {N_EPOCHS}")
print(f"  WARMUP_EPOCHS : {WARMUP_EPOCHS}")
print(f"  EARLY_STOP    : {EARLY_STOP}")
print(f"  ACCUM_STEPS   : {ACCUM_STEPS}")
print(f"  Optimizer     : AdamW  lr=3e-4  wd=1e-4")
print(f"  Scheduler     : CosineAnnealingLR  T_max={N_EPOCHS}")
print(f"\n✓ Ready")

Training config :
  N_EPOCHS      : 150
  WARMUP_EPOCHS : 15
  EARLY_STOP    : 25
  ACCUM_STEPS   : 4
  Optimizer     : AdamW  lr=3e-4  wd=1e-4
  Scheduler     : CosineAnnealingLR  T_max=150

✓ Ready


In [ ]:
model.train()
X_b, Y_b, _, _ = next(iter(train_loader))
X_b, Y_b = X_b.to(device), Y_b.to(device)
rho = X_b[:, 1:2, :, :]

with autocast('cuda', dtype=torch.float16):
    pred = model(X_b)
    loss, l_data, l_mass, l_bc = criterion(
        pred, Y_b, rho, phys_weight=0.0)

scaler.scale(loss).backward()
optimizer.zero_grad()

print(f"pred shape    : {pred.shape}")
print(f"L_data        : {l_data.item():.6f}")
print(f"Parameters    : {sum(p.numel() for p in model.parameters()):,}")
print(f"GPU allocated : {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"GPU reserved  : {torch.cuda.memory_reserved()/1e9:.2f} GB")
print(f"GPU free      : {(15.6-torch.cuda.memory_reserved()/1e9):.2f} GB")

torch.cuda.empty_cache()
print(f"\n✓ OOM check OK — safe to train")

pred shape    : torch.Size([1, 3, 500, 1000])
L_data        : 0.480972
Parameters    : 6,057,347
GPU allocated : 0.09 GB
GPU reserved  : 1.35 GB
GPU free      : 14.25 GB

✓ OOM check OK — safe to train


In [ ]:
def physics_warmup(epoch, warmup_epochs):
    if warmup_epochs <= 0:
        return 1.0
    return min(1.0, (epoch + 1) / warmup_epochs)


def compute_r2_per_channel(pred, true):
    pred_np = pred.detach().float().cpu().numpy()
    true_np = true.detach().float().cpu().numpy()
    scores  = []
    for c in range(pred_np.shape[1]):
        y_true = true_np[:, c].flatten()
        y_pred = pred_np[:, c].flatten()
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - y_true.mean()) ** 2)
        scores.append(1 - ss_res / (ss_tot + 1e-8))
    return np.mean(scores), scores


def train_epoch(epoch):
    model.train()
    train_loss = 0.0
    phys_w     = physics_warmup(epoch, WARMUP_EPOCHS)
    optimizer.zero_grad()

    for step, (X_b, Y_b, _, _) in enumerate(train_loader):
        X_b, Y_b = X_b.to(device), Y_b.to(device)
        rho = X_b[:, 1:2, :, :]

        with autocast('cuda', dtype=torch.float16):
            pred = model(X_b)
            loss, l_data, l_mass, l_bc = criterion(
                pred, Y_b, rho, phys_weight=phys_w)
            loss_scaled = loss / ACCUM_STEPS

        loss_scaled.backward()

        if (step+1) % ACCUM_STEPS == 0 or (step+1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        train_loss += loss.item()

    return train_loss / len(train_loader)


def validate_epoch(epoch):
    model.eval()
    val_loss      = 0.0
    r2_means      = []
    r2_per_ch_sum = np.zeros(3)
    phys_w        = physics_warmup(epoch, WARMUP_EPOCHS)

    with torch.no_grad():
        for X_b, Y_b, _, _ in val_loader:
            X_b, Y_b = X_b.to(device), Y_b.to(device)
            rho = X_b[:, 1:2, :, :]

            with autocast('cuda', dtype=torch.float16):
                pred = model(X_b)
                loss, _, _, _ = criterion(
                    pred, Y_b, rho, phys_weight=phys_w)

            val_loss += loss.item()
            r2_mean, r2_ch = compute_r2_per_channel(pred, Y_b)
            r2_means.append(r2_mean)
            r2_per_ch_sum += np.array(r2_ch)

    n = len(val_loader)
    return val_loss / n, np.mean(r2_means), (r2_per_ch_sum / n)


print("✓ physics_warmup / compute_r2 / train_epoch / validate_epoch defined")

✓ physics_warmup / compute_r2 / train_epoch / validate_epoch defined


In [ ]:
history = {
    'train_loss': [], 'val_loss': [], 'r2': [],
    'r2_u': [], 'r2_v': [], 'r2_w': [],
    'lr': [], 'phys_weight': []
}

best_val_loss    = float('inf')
best_epoch       = 0
best_r2          = -np.inf
patience_counter = 0

print("="*88)
print("STARTING VIT TRAINING")
print("="*88)
print(f"{'Ep':>4} | {'Train':>9} | {'Val':>9} | {'R²avg':>7} | "
      f"{'R²_u':>6} | {'R²_v':>6} | {'R²_w':>6} | "
      f"{'φ_w':>5} | {'LR':>8} | Status")
print("-"*88)

try:
    for epoch in range(N_EPOCHS):
        tr_loss            = train_epoch(epoch)
        va_loss, r2, r2_ch = validate_epoch(epoch)
        scheduler.step()

        lr     = optimizer.param_groups[0]['lr']
        phys_w = physics_warmup(epoch, WARMUP_EPOCHS)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['r2'].append(r2)
        history['r2_u'].append(r2_ch[0])
        history['r2_v'].append(r2_ch[1])
        history['r2_w'].append(r2_ch[2])
        history['lr'].append(lr)
        history['phys_weight'].append(phys_w)

        print(f"{epoch+1:>4} | {tr_loss:>9.5f} | {va_loss:>9.5f} | "
              f"{r2:>7.4f} | {r2_ch[0]:>6.4f} | {r2_ch[1]:>6.4f} | "
              f"{r2_ch[2]:>6.4f} | {phys_w:>5.2f} | {lr:>8.2e} | ", end="")

        if va_loss < best_val_loss:
            best_val_loss    = va_loss
            best_epoch       = epoch + 1
            best_r2          = r2
            patience_counter = 0
            torch.save({
                'epoch'               : epoch + 1,
                'model_state_dict'    : model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss'            : va_loss,
                'r2'                  : r2,
                'r2_u'                : r2_ch[0],
                'r2_v'                : r2_ch[1],
                'r2_w'                : r2_ch[2],
                'history'             : history,
            }, MODEL_DIR / 'best_vit_model.pth')
            print("✓ BEST")
        else:
            patience_counter += 1
            print(f"(patience {patience_counter}/{EARLY_STOP})"
                  if patience_counter % 5 == 0 else "")

        if patience_counter >= EARLY_STOP:
            print(f"\n⚠ Early stopping at epoch {epoch+1}")
            break

except KeyboardInterrupt:
    print("\n⚠ Training interrupted")
except Exception as e:
    print(f"\n✗ Training failed: {type(e).__name__}: {e}")
    import traceback; traceback.print_exc()

print("\n" + "="*88)
print("TRAINING COMPLETE")
print(f"Best epoch : {best_epoch}  "
      f"Best val_loss : {best_val_loss:.6f}  "
      f"Best R² : {best_r2:.4f}")
print("="*88)

pd.DataFrame(history).to_csv(
    OUT_DIR / 'vit_training_history.csv', index=False)
print(f"✓ History saved")

STARTING VIT TRAINING
  Ep |     Train |       Val |   R²avg |   R²_u |   R²_v |   R²_w |   φ_w |       LR | Status
----------------------------------------------------------------------------------------
   1 |   0.97180 |   0.95058 | -1.2192 | -1.5977 | -2.2781 | 0.2183 |  0.07 | 3.00e-04 | ✓ BEST
   2 |   0.92258 |   0.94526 | -1.1038 | -1.6692 | -1.8419 | 0.1998 |  0.13 | 3.00e-04 | ✓ BEST
   3 |   0.92949 |   0.91899 | -1.0503 | -1.6071 | -1.6533 | 0.1095 |  0.20 | 3.00e-04 | ✓ BEST
   4 |   0.89617 |   0.84654 | -0.8529 | -1.6116 | -1.1827 | 0.2357 |  0.27 | 2.99e-04 | ✓ BEST
   5 |   0.92726 |   0.90827 | -0.9823 | -1.6215 | -1.4867 | 0.1612 |  0.33 | 2.99e-04 | 
   6 |   1.00692 |   0.97661 | -1.2348 | -1.5969 | -2.2521 | 0.1446 |  0.40 | 2.99e-04 | 
   7 |   0.97940 |   0.89545 | -0.8913 | -1.6327 | -1.1812 | 0.1400 |  0.47 | 2.98e-04 | 
   8 |   0.88205 |   0.97750 | -0.9064 | -1.6068 | -1.0256 | -0.0868 |  0.53 | 2.98e-04 | 
   9 |   0.91104 |   0.82164 | -0.8711 | -2.0924 |

In [ ]:
ckpt = torch.load(MODEL_DIR / "best_vit_model.pth",
                  map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded epoch {ckpt['epoch']}  "
      f"val_loss={ckpt['val_loss']:.6f}  "
      f"R²avg={ckpt['r2']:.4f}  "
      f"R²_u={ckpt['r2_u']:.4f}  "
      f"R²_v={ckpt['r2_v']:.4f}  "
      f"R²_w={ckpt['r2_w']:.4f}")

results = []
with torch.no_grad():
    for X_b, Y_b, day, hour in test_loader:
        day, hour = day.item(), hour.item()
        with autocast('cuda', dtype=torch.float16):
            pred = model(X_b.to(device))
        pred_n = pred[0].float().cpu().numpy()
        true_n = Y_b[0].numpy()

        u_p = pred_n[0]*STD_U+MU_U; v_p = pred_n[1]*STD_V+MU_V
        w_p = pred_n[2]*STD_W+MU_W
        u_t = true_n[0]*STD_U+MU_U; v_t = true_n[1]*STD_V+MU_V
        w_t = true_n[2]*STD_W+MU_W

        mae_u  = np.mean(np.abs(u_p-u_t))
        mae_v  = np.mean(np.abs(v_p-v_t))
        mae_w  = np.mean(np.abs(w_p-w_t))
        rmse_u = np.sqrt(np.mean((u_p-u_t)**2))
        rmse_v = np.sqrt(np.mean((v_p-v_t)**2))
        rmse_w = np.sqrt(np.mean((w_p-w_t)**2))
        r2_u   = 1-np.sum((u_p-u_t)**2)/(np.sum((u_t-u_t.mean())**2)+1e-9)
        r2_v   = 1-np.sum((v_p-v_t)**2)/(np.sum((v_t-v_t.mean())**2)+1e-9)
        r2_w   = 1-np.sum((w_p-w_t)**2)/(np.sum((w_t-w_t.mean())**2)+1e-9)
        bias_u = np.mean(u_p-u_t)
        bias_v = np.mean(v_p-v_t)
        bias_w = np.mean(w_p-w_t)

        ws_p    = np.sqrt(u_p**2+v_p**2)
        ws_t    = np.sqrt(u_t**2+v_t**2)
        mae_ws  = np.mean(np.abs(ws_p-ws_t))
        dir_p   = np.degrees(np.arctan2(v_p,u_p))%360
        dir_t   = np.degrees(np.arctan2(v_t,u_t))%360
        dir_err = np.minimum(np.abs(dir_p-dir_t), 360-np.abs(dir_p-dir_t))
        mae_dir = np.mean(dir_err)

        acc_u_05 = (np.abs(u_p-u_t)<0.5).mean()*100
        acc_u_10 = (np.abs(u_p-u_t)<1.0).mean()*100
        acc_v_05 = (np.abs(v_p-v_t)<0.5).mean()*100
        acc_v_10 = (np.abs(v_p-v_t)<1.0).mean()*100
        acc_w_01 = (np.abs(w_p-w_t)<0.1).mean()*100

        results.append(dict(
            day=day, hour=hour,
            mae_u=mae_u, mae_v=mae_v, mae_w=mae_w,
            rmse_u=rmse_u, rmse_v=rmse_v, rmse_w=rmse_w,
            r2_u=r2_u, r2_v=r2_v, r2_w=r2_w,
            bias_u=bias_u, bias_v=bias_v, bias_w=bias_w,
            mae_ws=mae_ws, mae_dir=mae_dir,
            acc_u_05=acc_u_05, acc_u_10=acc_u_10,
            acc_v_05=acc_v_05, acc_v_10=acc_v_10,
            acc_w_01=acc_w_01,
        ))

        print(f"  J{day:3d} h{hour:02d} | "
              f"MAE u={mae_u:.4f} v={mae_v:.4f} w={mae_w:.4f} | "
              f"R² u={r2_u:.4f} v={r2_v:.4f} w={r2_w:.4f} | "
              f"Acc_u(±0.5)={acc_u_05:.1f}% "
              f"Acc_v(±0.5)={acc_v_05:.1f}%")

df_r = pd.DataFrame(results)
df_r.to_csv(OUT_DIR / 'vit_evaluation_results.csv', index=False)

print(f"\n{'='*72}")
print(f"VIT TEST SET METRICS (mean over {len(df_r)} snapshots):")
print(f"{'='*72}")
for col in ['mae_u','mae_v','mae_w','rmse_u','rmse_v','rmse_w',
            'r2_u','r2_v','r2_w','bias_u','bias_v','bias_w',
            'mae_ws','mae_dir',
            'acc_u_05','acc_u_10','acc_v_05','acc_v_10','acc_w_01']:
    print(f"  {col:12s}: {df_r[col].mean():.4f}")
print('='*72)
print(f"\n✓ Results saved: {OUT_DIR/'vit_evaluation_results.csv'}")

Loaded epoch 38  val_loss=0.958226  R²avg=-0.9446  R²_u=-1.6988  R²_v=-1.1913  R²_w=0.0563
  J 23 h00 | MAE u=2.3988 v=1.2236 w=0.0425 | R² u=-2.2988 v=-0.2662 w=0.0895 | Acc_u(±0.5)=9.5% Acc_v(±0.5)=22.5%
  J 23 h06 | MAE u=1.7819 v=1.2094 w=0.0430 | R² u=-1.6907 v=0.0386 w=0.0396 | Acc_u(±0.5)=16.1% Acc_v(±0.5)=24.5%
  J 23 h12 | MAE u=3.6170 v=1.2606 w=0.0532 | R² u=-3.3082 v=0.0381 w=0.2068 | Acc_u(±0.5)=3.5% Acc_v(±0.5)=23.5%
  J 23 h18 | MAE u=2.8998 v=1.5986 w=0.0471 | R² u=-2.6959 v=-2.0698 w=0.2040 | Acc_u(±0.5)=6.6% Acc_v(±0.5)=11.4%
  J 26 h00 | MAE u=1.0909 v=1.3622 w=0.0374 | R² u=-7.9296 v=-0.0373 w=-0.1888 | Acc_u(±0.5)=8.6% Acc_v(±0.5)=23.1%
  J 26 h06 | MAE u=1.8739 v=1.3865 w=0.0364 | R² u=-33.3521 v=-0.0594 w=-0.3250 | Acc_u(±0.5)=0.1% Acc_v(±0.5)=22.9%
  J 26 h12 | MAE u=0.8903 v=2.2610 w=0.0456 | R² u=-2.4838 v=-0.7365 w=0.0500 | Acc_u(±0.5)=19.2% Acc_v(±0.5)=16.9%
  J 26 h18 | MAE u=0.5902 v=1.2883 w=0.0406 | R² u=-0.3697 v=0.0120 w=0.0696 | Acc_u(±0.5)=47.0% Acc_